# 03-1 · The satisfaction matrix S

| | |
|---|---|
| **What it does** | scores every member against every option in F: `s_i(x) = cosine(taste_i, taste_x)` |
| **Proves** | the input every selector in 03-2 reduces to one number |
| **Ported from** | `nbio.taste_similarity` / `nbio.calculate_cosine_similarity` — Vincent's `resolve_taste_vector` (potluck, `services/taste_service.py`), the cosine itself ported from `integrations/embeddings.py` |
| **Eval surface** | the range of S — watch how compressed it is |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## Rebuild the group and F from 02-1


In [3]:
# Step 1: the same 4-member group and feasible set as 02-safety/01
personas = json.load(open(nbio.DATA / "personas.json"))
group_names = ["Vihaan", "Vincent", "Vasu", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"],
                     "taste": p["taste_vector"]})

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip()
            return
        for v in node.values():
            yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node:
            yield from iter_dish_names(v)

import csv
flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        name = row.get("ingredient", "").strip().lower()
        if name:
            flavor_map[name] = {d: float(row.get(d, 0) or 0) for d in nbio.TASTE_DIMS}

def infer_taste(dish_name):
    matched = [flavor_map[ing] for ing in flavor_map if ing in dish_name.lower()]
    return [sum(m[d] for m in matched) / len(matched) for d in nbio.TASTE_DIMS] if matched else [0.0]*6

catalog = []
for f in sorted((nbio.DATA / "Menu").glob("*.json")):
    payload = json.loads(f.read_text())
    names = [n for n in iter_dish_names(payload) if n]
    vecs = [infer_taste(n) for n in names]
    nz = [v for v in vecs if sum(abs(x) for x in v) > 0]
    avg = [sum(v[i] for v in nz)/len(nz) for i in range(6)] if nz else [0.0]*6
    catalog.append({"option_id": f.stem, "name": f.stem,
                     "menu": [{"name": n} for n in names], "taste": avg})

F = nbio.build_feasible_set(catalog, members)
print(f"members: {[m['display_name'] for m in members]}  |  F size: {len(F)}")


members: ['Vihaan', 'Vincent', 'Vasu', 'Alice']  |  F size: 5


## Score every member against every option — Vincent's `member_scores_for_candidate`

`nbio.taste_similarity(member_taste, option_taste)` per member, per option. The
grid this builds is S.


In [4]:
# Step 2: build S
member_ids = [m["member_id"] for m in members]
names = {m["member_id"]: m["display_name"] for m in members}
labels = {o["option_id"]: o["name"] for o in F}

S = {}
for o in F:
    S[o["option_id"]] = {m["member_id"]: nbio.taste_similarity(m["taste"], o["taste"]) for m in members}

before = {"F": len(F)}
nbio.io(2, "satisfaction_matrix", before, {"F": len(F), "S": len(S)},
        reads=["F (option taste)", "members (taste)"])
nbio.matrix(S, member_ids, names=names, labels=labels)

vals = [v for row in S.values() for v in row.values()]
print(f"\nrange {min(vals):.3f} .. {max(vals):.3f}   spread {max(vals)-min(vals):.3f}")


+- STEP 2  satisfaction_matrix                                      [py]
| IN      : F (option taste), members (taste)
| NEW     : S
| CHANGED : -
+-------------------------------------------------------------------------
option                            Vihaan   Vincent      Vasu     Alice      min      mean
-----------------------------------------------------------------------------------------
Arabian Corner Restaurant -...     0.501     0.633     0.709     0.787    0.501    0.658
Casa.cafe                          0.624     0.746     0.825     0.849    0.624    0.761
Delicia Multi Cuisine Resta...     0.622     0.747     0.834     0.833    0.622    0.759
House of Chefs                     0.540     0.704     0.774     0.801    0.540    0.705
Ministry of South Bar and K...     0.529     0.670     0.755     0.801    0.529    0.689

range 0.501 .. 0.849   spread 0.348


### Reading it

Cosine similarity over non-negative 6-D vectors cannot go low — with no negative
components every score lands in roughly 0.5-1.0. "Satisfaction" therefore has far
less dynamic range than a 0-1 scale suggests; **differences between options
matter more than levels**. Carry `S`, `members`, and `F` forward — **03-2** turns
this grid into a single pick.
